# 07 — Fine-tuning ConvNeXt-Tiny (LLRD + High Resolution)

Studio Proyek 2 — Klasifikasi 5 jenis sampah (IF25-40401)

## 1. Setup

Semua import, path, dan pengecekan GPU. Dibanding notebook 06 ada tambahan `param_groups_layer_decay` dari timm untuk *layer-wise LR decay*.

In [1]:
import gc
import json
import math
import os
import random
import shutil
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn as nn
from huggingface_hub.utils import logging as hf_logging
from PIL import Image
from timm.optim import param_groups_layer_decay
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import evaluate
from evaluate import metrics
from preprocessing import MEAN, SEED, STD, get_batch_mix, get_eval_transform, get_train_transform

TRAIN_DIR, VAL_DIR = ROOT / "my_train", ROOT / "my_val"
SUB_DIR, CKPT_DIR = ROOT / "submission", ROOT / "checkpoints"
CKPT_DIR.mkdir(exist_ok=True)
CLASSES = ["botol_plastik", "gelas_plastik", "kertas_kardus", "saset_kemasan", "styrofoam"]
NUM_CLASSES = len(CLASSES)
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
NUM_WORKERS = 0

hf_logging.set_verbosity_error()

assert torch.cuda.is_available(), "notebook ini butuh GPU CUDA"
device = "cuda"
torch.backends.cudnn.benchmark = True

gpu = torch.cuda.get_device_properties(0)
print(f"torch {torch.__version__} | timm {timm.__version__}")
print(f"GPU   : {gpu.name}, {gpu.total_memory / 1024**3:.1f} GB")

torch 2.11.0+cu128 | timm 1.0.30
GPU   : NVIDIA GeForce RTX 3050 Laptop GPU, 4.0 GB


## 2. Verifikasi dataset

Memakai `my_train/` dan `my_val/` hasil `03_preprocessing.ipynb`, sama dengan notebook 06.

In [2]:
for d in (TRAIN_DIR, VAL_DIR):
    found = sorted(p.name for p in d.iterdir() if p.is_dir())
    assert found == CLASSES, f"folder kelas di {d} tidak sesuai: {found}"

corrupt = []
print(f"{'kelas':<15} {'train':>6} {'val':>6}")
for c in CLASSES:
    n = {}
    for split, d in (("train", TRAIN_DIR), ("val", VAL_DIR)):
        files = [p for p in (d / c).iterdir() if p.suffix.lower() in IMG_EXTS]
        n[split] = len(files)
        for p in files:
            try:
                with Image.open(p) as im:
                    im.verify()
            except Exception as e:
                corrupt.append((p, repr(e)))
    print(f"{c:<15} {n['train']:>6} {n['val']:>6}")
assert not corrupt, f"ada gambar rusak: {corrupt[:3]}"
print("semua gambar valid.")

kelas            train    val
botol_plastik      445     78
gelas_plastik      443     78
kertas_kardus      311     54
saset_kemasan      573    100
styrofoam          571    100
semua gambar valid.


## 3. Konfigurasi

Resep notebook 06 ditambah *stochastic depth*, *layer-wise LR decay*, dan fase singkat 3 epoch di 288 px.

In [3]:
CFG = dict(
    arch="convnext_tiny.fb_in22k_ft_in1k",
    img_size=224,
    batch=16,
    accum=2,             # grad accumulation -> batch efektif 32
    drop_path=0.1,       # baru: stochastic depth
    layer_decay=0.85,    # baru: LR tiap layer = LR layer di atasnya x 0.85
    lr=1e-4,             # LR layer teratas di fase penuh, turun x0.85 per layer ke bawah
    head_lr_mult=10,     # LR head = lr x 10
    wd=5e-2,
    head_epochs=2,       # fase head: backbone dibekukan
    head_lr=1e-3,
    epochs=20,           # fase penuh, sama dengan nb06
    warmup_epochs=1,
    hr_size=288,         # baru: fase resolusi tinggi
    hr_epochs=3,
    hr_lr=2e-5,          # LR layer teratas di fase resolusi tinggi (konstan)
    hr_ema_decay=0.99,   # langkahnya sedikit -> EMA lebih cepat mengikuti
    label_smoothing=0.1,
    mix_p=0.5,           # peluang Mixup/CutMix per batch
    ema_decay=0.995,
)
RETRAIN = False  # True = latih ulang cell 8-9 walaupun checkpoint-nya sudah ada di checkpoints/
for k, v in CFG.items():
    print(f"{k:<16} {v}")

arch             convnext_tiny.fb_in22k_ft_in1k
img_size         224
batch            16
accum            2
drop_path        0.1
layer_decay      0.85
lr               0.0001
head_lr_mult     10
wd               0.05
head_epochs      2
head_lr          0.001
epochs           20
warmup_epochs    1
hr_size          288
hr_epochs        3
hr_lr            2e-05
hr_ema_decay     0.99
label_smoothing  0.1
mix_p            0.5
ema_decay        0.995


## 4. Uji resolusi tanpa training

Submission sekarang (dilatih di 224) dievaluasi di beberapa resolusi, dengan dan tanpa center crop.

In [4]:
kw = dict(num_workers=NUM_WORKERS, pin_memory=True)


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        with torch.autocast("cuda", dtype=torch.float16):
            ps += model(x).argmax(1).cpu().tolist()
        ys += y.tolist()
    return metrics(ys, ps)


def val_loader(tf):
    return DataLoader(datasets.ImageFolder(VAL_DIR, transform=tf), batch_size=32, shuffle=False, **kw)


def square_transform(size):
    # Alternatif tanpa crop: gambar lebar tidak terpotong, tapi rasio aspek berubah.
    return transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])


sub_model, _ = evaluate.load_submission(SUB_DIR, device)
sub_model.to(memory_format=torch.channels_last)
print(f"{'transform val':<20} {'Macro-F1':>9} {'akurasi':>8} {'F1 saset':>9}")
for s in (224, 256, 288, 320):
    for name, tf in ((f"resize+crop {s}", get_eval_transform(s)), (f"persegi {s}", square_transform(s))):
        m = evaluate_model(sub_model, val_loader(tf))
        print(f"{name:<20} {m['macro_f1']:>9.4f} {m['accuracy']:>8.4f} {m['per_class_f1']['saset_kemasan']:>9.4f}")
del sub_model
gc.collect()
torch.cuda.empty_cache()

transform val         Macro-F1  akurasi  F1 saset
resize+crop 224         0.9774   0.9780    0.9583
persegi 224             0.9587   0.9634    0.9534
resize+crop 256         0.9767   0.9780    0.9637
persegi 256             0.9660   0.9683    0.9474
resize+crop 288         0.9611   0.9634    0.9479
persegi 288             0.9573   0.9610    0.9529
resize+crop 320         0.9639   0.9659    0.9588
persegi 320             0.9500   0.9537    0.9375


## 5. Transform & DataLoader

Dua set loader: 224 untuk fase head dan fase penuh, 288 untuk fase resolusi tinggi. Val tetap `Resize + CenterCrop`

In [5]:
train_ds = datasets.ImageFolder(TRAIN_DIR, transform=get_train_transform(CFG["img_size"], strong=True))
train_ds_hr = datasets.ImageFolder(TRAIN_DIR, transform=get_train_transform(CFG["hr_size"], strong=True))
val_ds = datasets.ImageFolder(VAL_DIR, transform=get_eval_transform(CFG["img_size"]))
val_ds_hr = datasets.ImageFolder(VAL_DIR, transform=get_eval_transform(CFG["hr_size"]))
for d in (train_ds, train_ds_hr, val_ds, val_ds_hr):
    assert d.classes == CLASSES, "urutan kelas tidak sesuai kontrak submission!"

train_dl = DataLoader(train_ds, batch_size=CFG["batch"], shuffle=True, drop_last=True, **kw)
train_dl_hr = DataLoader(train_ds_hr, batch_size=CFG["batch"], shuffle=True, drop_last=True, **kw)
val_dl = DataLoader(val_ds, batch_size=32, shuffle=False, **kw)
val_dl_hr = DataLoader(val_ds_hr, batch_size=32, shuffle=False, **kw)

print(f"train: {len(train_ds)} gambar, {len(train_dl)} batch | val: {len(val_ds)} gambar, {len(val_dl)} batch")
for name, loader in (("train 224", train_dl), ("train 288", train_dl_hr), ("val 224", val_dl), ("val 288", val_dl_hr)):
    x, _ = next(iter(loader))
    print(f"{name:<10} batch {tuple(x.shape)}")

train: 2343 gambar, 146 batch | val: 410 gambar, 13 batch
train 224  batch (16, 3, 224, 224)
train 288  batch (16, 3, 288, 288)
val 224    batch (32, 3, 224, 224)
val 288    batch (32, 3, 288, 288)


## 6. Model + layer-wise LR decay

ConvNeXt-Tiny pretrained ImageNet-21k dengan *stochastic depth* 0,1. LR mengecil dari head ke stem, jadi fitur umum di layer awal hampir tidak berubah dan layer akhir lebih bebas beradaptasi ke 5 kelas sampah.

In [6]:
def build_model(pretrained=True):
    # drop_path tidak punya parameter, jadi state_dict tetap cocok dengan get_model() submission.
    return timm.create_model(CFG["arch"], pretrained=pretrained, num_classes=NUM_CLASSES,
                             drop_path_rate=CFG["drop_path"])


def llrd_groups(model, base_lr):
    # Layer-wise LR decay dari timm: lr_scale = layer_decay^(jarak layer ke head), bias/norm tanpa weight decay.
    # head.fc (layer baru, acak) dipisah supaya dapat LR x head_lr_mult; head.norm tetap pretrained.
    fc_ids = {id(p) for p in model.get_classifier().parameters()}
    groups = []
    for g in param_groups_layer_decay(model, weight_decay=CFG["wd"], layer_decay=CFG["layer_decay"]):
        for is_fc in (False, True):
            params = [p for p in g["params"] if (id(p) in fc_ids) == is_fc]
            if params:
                mult = CFG["head_lr_mult"] if is_fc else 1
                groups.append(dict(params=params, weight_decay=g["weight_decay"], lr=base_lr * g["lr_scale"] * mult))
    return groups


_m = build_model()
names = {id(p): n for n, p in _m.named_parameters()}
rows = {}
for g in llrd_groups(_m, CFG["lr"]):
    for p in g["params"]:
        parts = names[id(p)].split(".")
        key = parts[0] if parts[0] == "stem" else ".".join(parts[:2])  # stem / stages.i / head.norm / head.fc
        r = rows.setdefault(key, dict(n=0, lrs=set()))
        r["n"] += p.numel()
        r["lrs"].add(g["lr"])
assert sum(r["n"] for r in rows.values()) == sum(p.numel() for p in _m.parameters()), "ada parameter yang tidak masuk group"

print(f"{'bagian':<10} {'parameter':>10}  LR (fase penuh)")
for key, r in rows.items():
    lo, hi = min(r["lrs"]), max(r["lrs"])
    lr_txt = f"{lo:.2e}" if lo == hi else f"{lo:.2e} - {hi:.2e}"
    print(f"{key:<10} {r['n'] / 1e6:>9.2f}M  {lr_txt}")
print("drop path blok terakhir:", _m.stages[3].blocks[2].drop_path)
del _m, names, rows

bagian      parameter  LR (fase penuh)
stem            0.00M  4.56e-06
stages.0        0.24M  5.36e-06 - 7.43e-06
stages.1        0.99M  8.74e-06 - 1.21e-05
stages.2       11.11M  1.42e-05 - 5.22e-05
stages.3       15.47M  6.14e-05 - 8.50e-05
head.norm       0.00M  1.00e-04
head.fc         0.00M  1.00e-03
drop path blok terakhir: DropPath(drop_prob=0.100)


## 7. Fungsi training

Sama dengan notebook 06 (AMP, grad accumulation, EMA, warmup + cosine). `run_epoch` menerima `loader` supaya bisa dipakai di 224 maupun 288, dan `log_epoch` mencatat skor serta menyimpan bobot terbaik.

In [7]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def warmup_cosine(total_steps, warmup_steps):
    def f(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        t = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * t))
    return f


def run_epoch(model, opt, loader, sched=None, ema=None, use_mix=True):
    # Sama dengan nb06, tapi loader jadi argumen (224 atau 288) dan Mixup/CutMix bisa dimatikan.
    model.train()
    total, n = 0.0, 0
    opt.zero_grad(set_to_none=True)
    for i, (x, y) in enumerate(loader):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        if use_mix:
            x, y = mix(x, y)  # Mixup/CutMix untuk sebagian batch (label jadi lunak)
        x = x.to(memory_format=torch.channels_last)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = loss_fn(model(x), y)
        scaler.scale(loss / CFG["accum"]).backward()
        if (i + 1) % CFG["accum"] == 0 or i + 1 == len(loader):
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            if sched is not None:
                sched.step()
            if ema is not None:
                ema.update_parameters(model)
        total += loss.item() * len(x)
        n += len(x)
    return total / n


def log_epoch(phase, ep, loss, size, model, m, m_ema=None, ema=None, t0=None):
    # Catat epoch ke `history`; kalau Macro-F1 (raw atau EMA) lebih tinggi dari `best`, salin bobotnya ke CPU.
    history.append(dict(phase=phase, epoch=ep, size=size, loss=loss, f1=m["macro_f1"], acc=m["accuracy"],
                        f1_ema=m_ema["macro_f1"] if m_ema else None))
    flag = ""
    for src, mm, net in (("raw", m, model), ("ema", m_ema, ema.module if ema else None)):
        if mm and mm["macro_f1"] > best["f1"]:
            best.update(f1=mm["macro_f1"], src=src, epoch=f"{phase}{ep}", size=size,
                        state={k: v.detach().cpu().clone() for k, v in net.state_dict().items()})
            flag = f"  <- disimpan ({src})"
    loss_txt = f"loss {loss:.4f}" if loss is not None else "loss    -  "
    ema_txt = f" | EMA {m_ema['macro_f1']:.4f}" if m_ema else ""
    t_txt = f"{time.time() - t0:.0f}s, " if t0 else ""
    print(f"[{phase}] ep {ep:2d}  {loss_txt} | F1 {m['macro_f1']:.4f} acc {m['accuracy']:.4f}{ema_txt} | "
          f"{t_txt}GPU {torch.cuda.max_memory_allocated() / 1024**3:.1f} GB{flag}")

## 8. Training fase head dan penuh (224)

Fase head (backbone beku), lalu fase penuh dengan LLRD + EMA. Bobot terbaik dan bobot EMA akhir disimpan ke `checkpoints/` supaya fase 288 tidak bergantung pada memori kernel.

In [8]:
if not RETRAIN and (CKPT_DIR / "convnext_t_nb07_llrd_224_best.pth").exists():
    print("checkpoint fase 224 sudah ada -> training dilewati (set RETRAIN = True di cell 3 untuk melatih ulang)")
else:
    set_seed(SEED)
    model = build_model().to(device, memory_format=torch.channels_last)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])
    mix = get_batch_mix(NUM_CLASSES, p=CFG["mix_p"])
    scaler = torch.amp.GradScaler("cuda")
    torch.cuda.reset_peak_memory_stats()

    best = dict(f1=-1.0, src=None, epoch=None, size=None, state=None)
    history = []
    t_start = time.time()

    # Fase 1 - head: backbone dibekukan, hanya head.fc baru yang dilatih
    head_ids = {id(p) for p in model.get_classifier().parameters()}
    for p in model.parameters():
        p.requires_grad = id(p) in head_ids
    opt = torch.optim.AdamW(model.get_classifier().parameters(), lr=CFG["head_lr"], weight_decay=CFG["wd"])
    for ep in range(1, CFG["head_epochs"] + 1):
        t0 = time.time()
        loss = run_epoch(model, opt, train_dl)
        log_epoch("head", ep, loss, CFG["img_size"], model, evaluate_model(model, val_dl), t0=t0)

    # Fase 2 - penuh: semua layer dengan LLRD, warmup + cosine per langkah optimizer, EMA bobot
    for p in model.parameters():
        p.requires_grad = True
    opt = torch.optim.AdamW(llrd_groups(model, CFG["lr"]))
    steps_per_epoch = math.ceil(len(train_dl) / CFG["accum"])
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, warmup_cosine(steps_per_epoch * CFG["epochs"], steps_per_epoch * CFG["warmup_epochs"]))
    ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(CFG["ema_decay"]), use_buffers=True)
    for ep in range(1, CFG["epochs"] + 1):
        t0 = time.time()
        loss = run_epoch(model, opt, train_dl, sched, ema)
        log_epoch("full", ep, loss, CFG["img_size"], model, evaluate_model(model, val_dl),
                  evaluate_model(ema.module, val_dl), ema, t0)

    # Simpan ke disk supaya fase 288 (cell 9) bisa dijalankan terpisah, juga setelah kernel di-restart
    torch.save(best["state"], CKPT_DIR / "convnext_t_nb07_llrd_224_best.pth")
    torch.save(ema.module.state_dict(), CKPT_DIR / "convnext_t_nb07_llrd_224_ema.pth")
    log224 = dict(cfg=CFG, best_f1=best["f1"], best_src=best["src"], best_epoch=best["epoch"], best_size=best["size"],
                  minutes=round((time.time() - t_start) / 60, 1), history=history)
    (CKPT_DIR / "convnext_t_nb07_llrd_224.json").write_text(json.dumps(log224, indent=1), encoding="utf-8")
    print()
    print(f"Fase 224 selesai: Macro-F1 terbaik {best['f1']:.4f} ({best['src']}, {best['epoch']}), "
          f"{log224['minutes']:.1f} menit -> checkpoints/convnext_t_nb07_llrd_224_*.pth")

checkpoint fase 224 sudah ada -> training dilewati (set RETRAIN = True di cell 3 untuk melatih ulang)


## 9. Fase resolusi tinggi (288)

Tiga epoch singkat di 288 px dari bobot EMA akhir fase 224, LR kecil konstan, epoch terakhir tanpa Mixup/CutMix. Checkpoint terbaik dipilih dari semua fase, dan resolusinya menentukan `get_transform()` submission.

In [9]:
if not RETRAIN and (CKPT_DIR / "convnext_t_nb07_llrd.pth").exists():
    print("checkpoint fase 288 sudah ada -> training dilewati (set RETRAIN = True di cell 3 untuk melatih ulang)")
else:
    # Fase 3 - resolusi tinggi. Dibaca dari disk (hasil cell 8), jadi cell ini aman dijalankan setelah restart kernel.
    set_seed(SEED)
    gc.collect()
    torch.cuda.empty_cache()
    need = [CKPT_DIR / f"convnext_t_nb07_llrd_224{x}" for x in (".json", "_best.pth", "_ema.pth")]
    missing = [p.name for p in need if not p.exists()]
    assert not missing, f"jalankan cell 8 (fase 224) sampai selesai dulu, belum ada: {missing}"
    log224 = json.loads((CKPT_DIR / "convnext_t_nb07_llrd_224.json").read_text(encoding="utf-8"))
    history = log224["history"]
    best = dict(f1=log224["best_f1"], src=log224["best_src"], epoch=log224["best_epoch"], size=log224["best_size"],
                state=torch.load(CKPT_DIR / "convnext_t_nb07_llrd_224_best.pth", map_location="cpu", weights_only=True))

    model = build_model(pretrained=False)
    model.load_state_dict(torch.load(CKPT_DIR / "convnext_t_nb07_llrd_224_ema.pth", map_location="cpu", weights_only=True))
    model.to(device, memory_format=torch.channels_last)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])
    mix = get_batch_mix(NUM_CLASSES, p=CFG["mix_p"])
    scaler = torch.amp.GradScaler("cuda")
    torch.cuda.reset_peak_memory_stats()
    t_start = time.time()

    # Epoch 0 = bobot EMA akhir fase 224 dinilai langsung di 288, tanpa training
    log_epoch("hr", 0, None, CFG["hr_size"], model, evaluate_model(model, val_dl_hr))
    opt = torch.optim.AdamW(llrd_groups(model, CFG["hr_lr"]))
    ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(CFG["hr_ema_decay"]), use_buffers=True)
    for ep in range(1, CFG["hr_epochs"] + 1):
        t0 = time.time()
        loss = run_epoch(model, opt, train_dl_hr, ema=ema, use_mix=ep < CFG["hr_epochs"])  # epoch terakhir tanpa Mixup/CutMix
        log_epoch("hr", ep, loss, CFG["hr_size"], model, evaluate_model(model, val_dl_hr),
                  evaluate_model(ema.module, val_dl_hr), ema, t0)

    minutes = log224["minutes"] + (time.time() - t_start) / 60
    torch.save(best["state"], CKPT_DIR / "convnext_t_nb07_llrd.pth")
    log = dict(name="convnext_t_llrd", seed=SEED, cfg=CFG, best_f1=best["f1"], best_src=best["src"],
               best_epoch=best["epoch"], best_size=best["size"], minutes=round(minutes, 1),
               peak_gpu_gb=round(torch.cuda.max_memory_allocated() / 1024**3, 2), history=history)
    (CKPT_DIR / "convnext_t_nb07_llrd.json").write_text(json.dumps(log, indent=1), encoding="utf-8")
    print()
    print(f"Macro-F1 val terbaik: {best['f1']:.4f} ({best['src']}, epoch {best['epoch']}, {best['size']} px), "
          f"{minutes:.1f} menit -> checkpoints/convnext_t_nb07_llrd.pth")

checkpoint fase 288 sudah ada -> training dilewati (set RETRAIN = True di cell 3 untuk melatih ulang)


## 10. Model soup

Bobot beberapa checkpoint yang di-fine-tune dari pretrained yang sama dirata-rata menjadi satu model (*uniform soup*).

In [10]:
# Lepas memori training fase 288
model = ema = opt = None
gc.collect()
torch.cuda.empty_cache()

# Semua checkpoint di-fine-tune dari bobot pretrained dan seed yang sama, jadi bobotnya bisa dirata-rata.
# convnext_t_full.pth tidak dipakai karena ikut dilatih dengan my_val.
CKPTS = {
    "submission (nb06)": SUB_DIR / "model.pth",
    "nb06 run ulang": CKPT_DIR / "convnext_t_nb07.pth",
    "nb07 LLRD best": CKPT_DIR / "convnext_t_nb07_llrd_224_best.pth",
    "nb07 LLRD EMA": CKPT_DIR / "convnext_t_nb07_llrd_224_ema.pth",
}
sds = {}
for name, path in CKPTS.items():
    sd = torch.load(path, map_location="cpu", weights_only=True)
    sds[name] = {k: v.float() if v.is_floating_point() else v for k, v in sd.items()}

# Gambar val di-decode sekali lalu disimpan di GPU (fp16), supaya tiap model cukup forward saja
xs, ys = [], []
for x, y in val_dl:
    xs.append(x.half().to(device).to(memory_format=torch.channels_last))
    ys += y.tolist()
net = build_model(pretrained=False).to(device, memory_format=torch.channels_last).eval()


@torch.no_grad()
def val_probs(sd):
    net.load_state_dict(sd)
    out = []
    for x in xs:
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(net(x).float().softmax(1).cpu())
    return torch.cat(out)


def make_soup(names):
    # Uniform soup: rata-rata semua bobot float; buffer integer diambil dari model pertama.
    first = sds[names[0]]
    return {k: sum(sds[n][k] for n in names) / len(names) if v.is_floating_point() else v for k, v in first.items()}


probs = {name: val_probs(sd) for name, sd in sds.items()}
names = list(CKPTS)
combos = [[n] for n in names] + [names[:2], [names[0], names[2]], names[:3], names]

print(f"{'kombinasi':<62} {'soup F1':>8} {'ensemble F1':>12}")
for combo in combos:
    label = " + ".join(combo)
    if len(combo) == 1:
        f1 = metrics(ys, probs[combo[0]].argmax(1).tolist())["macro_f1"]
        print(f"{label:<62} {f1:>8.4f} {'-':>12}")
        continue
    f1_soup = metrics(ys, val_probs(make_soup(combo)).argmax(1).tolist())["macro_f1"]
    f1_ens = metrics(ys, torch.stack([probs[n] for n in combo]).mean(0).argmax(1).tolist())["macro_f1"]
    print(f"{label:<62} {f1_soup:>8.4f} {f1_ens:>12.4f}")

kombinasi                                                       soup F1  ensemble F1
submission (nb06)                                                0.9774            -
nb06 run ulang                                                   0.9744            -
nb07 LLRD best                                                   0.9739            -
nb07 LLRD EMA                                                    0.9708            -
submission (nb06) + nb06 run ulang                               0.9759       0.9760
submission (nb06) + nb07 LLRD best                               0.9780       0.9844
submission (nb06) + nb06 run ulang + nb07 LLRD best              0.9790       0.9790
submission (nb06) + nb06 run ulang + nb07 LLRD best + nb07 LLRD EMA   0.9768       0.9768


## 11. Tulis submission ensemble

Dua ConvNeXt-Tiny dibungkus dalam satu `nn.Module` yang merata-rata probabilitas, ditulis dulu ke `checkpoints/submission_ensemble/`, dan baru disalin ke `submission/` kalau skornya lebih tinggi.

In [11]:
MODEL_PY = '''# Submission Studio 2 - ensemble 2 ConvNeXt-Tiny (timm convnext_tiny.fb_in22k_ft_in1k).
# Anggota 0: resep notebook 06. Anggota 1: resep notebook 07 (layer-wise LR decay + drop path).

import timm
import torch
import torch.nn as nn
from torchvision import transforms

NUM_CLASSES = 5
ARCH = "convnext_tiny.fb_in22k_ft_in1k"
N_MEMBERS = 2


class Ensemble(nn.Module):
    def __init__(self):
        super().__init__()
        self.members = nn.ModuleList(
            timm.create_model(ARCH, pretrained=False, num_classes=NUM_CLASSES) for _ in range(N_MEMBERS))

    def forward(self, x):
        # Rata-rata probabilitas anggota; log() supaya output tetap logits [batch, 5] dan argmax tidak berubah.
        probs = torch.stack([m(x).softmax(1) for m in self.members]).mean(0)
        return probs.clamp_min(1e-12).log()


def get_model():
    # Arsitektur saja: tanpa mengunduh bobot (evaluate.py yang memuat model.pth).
    return Ensemble()


def get_transform():
    # Kedua anggota dilatih dengan transform validasi yang sama (get_eval_transform(224)).
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
'''

# Lepas memori cell 10
net = xs = sds = probs = None
gc.collect()
torch.cuda.empty_cache()

# Cadangkan submission satu model (nb06, 0,9774) sekali saja; anggota 0 diambil dari cadangan ini
BACKUP, NEW_DIR = CKPT_DIR / "submission_nb06", CKPT_DIR / "submission_ensemble"
if not BACKUP.exists():
    shutil.copytree(SUB_DIR, BACKUP, ignore=shutil.ignore_patterns("__pycache__"))
assert "Ensemble" not in (BACKUP / "model.py").read_text(encoding="utf-8"), "cadangan bukan submission satu model"
members = [BACKUP / "model.pth", CKPT_DIR / "convnext_t_nb07_llrd_224_best.pth"]

# Gabungkan bobot: kunci "members.<i>.<kunci asli>", disimpan fp16 seperti submission sebelumnya
state = {}
for i, path in enumerate(members):
    for k, v in torch.load(path, map_location="cpu", weights_only=True).items():
        state[f"members.{i}.{k}"] = v.half() if v.is_floating_point() else v
NEW_DIR.mkdir(exist_ok=True)
(NEW_DIR / "model.py").write_text(MODEL_PY, encoding="utf-8")
(NEW_DIR / "requirements.txt").write_text("timm\n", encoding="utf-8")
torch.save(state, NEW_DIR / "model.pth")
del state

# Nilai keduanya dengan loader resmi evaluate.py (load_state_dict strict=True)
scores = {}
for name, d in (("submission sekarang", SUB_DIR), ("ensemble baru", NEW_DIR)):
    sub_model, sub_tf = evaluate.load_submission(d, device)
    scores[name] = evaluate_model(sub_model.to(memory_format=torch.channels_last), val_loader(sub_tf))["macro_f1"]
    del sub_model
    torch.cuda.empty_cache()
    print(f"{name:<20} Macro-F1 {scores[name]:.4f}")
print(f"model.pth ensemble: {(NEW_DIR / 'model.pth').stat().st_size / 1e6:.1f} MB")

_, new_tf = evaluate.load_submission(NEW_DIR, "cpu")
with Image.open(next(VAL_DIR.rglob("*.jpg"))) as im:
    im = im.convert("RGB")
    assert torch.equal(new_tf(im), get_eval_transform(CFG["img_size"])(im)), "get_transform() tidak sama!"

if scores["ensemble baru"] <= scores["submission sekarang"]:
    print("ensemble tidak lebih baik -> submission tidak diubah")
else:
    for f in ("model.py", "model.pth", "requirements.txt"):
        shutil.copy2(NEW_DIR / f, SUB_DIR / f)
    print(f"submission diganti dengan ensemble (cadangan satu model: {BACKUP})")

submission sekarang  Macro-F1 0.9774
ensemble baru        Macro-F1 0.9844
model.pth ensemble: 111.4 MB
submission diganti dengan ensemble (cadangan satu model: d:\Kampus\SEMESTER 7\Pembelajaran Mendalam\Studio_Proyek2\checkpoints\submission_nb06)


## 12. Uji dengan `evaluate.py`

Skrip resmi dijalankan di kernel ini (fp32, tanpa AMP).

In [12]:
gc.collect()
torch.cuda.empty_cache()

_argv, t0 = sys.argv, time.time()
sys.argv = ["evaluate.py", "--submission", str(SUB_DIR), "--data", str(VAL_DIR)]
try:
    evaluate.main()
finally:
    sys.argv = _argv
print(f"\nwaktu evaluasi: {time.time() - t0:.0f} detik")

device=cuda  gambar=410

Macro-F1 : 0.9844
Akurasi  : 0.9854
  F1 botol_plastik   0.9873
  F1 gelas_plastik   0.9872
  F1 kertas_kardus   0.9730
  F1 saset_kemasan   0.9744
  F1 styrofoam       1.0000
Confusion matrix (baris = label asli, kolom = prediksi):
  botol_plastik   [78, 0, 0, 0, 0]
  gelas_plastik   [1, 77, 0, 0, 0]
  kertas_kardus   [0, 0, 54, 0, 0]
  saset_kemasan   [1, 1, 3, 95, 0]
  styrofoam       [0, 0, 0, 0, 100]

waktu evaluasi: 32 detik
